In [20]:
import requests
import time
import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
from google.colab import userdata

API_KEY = userdata.get('OPENAQ_API_KEY').strip()
assert len(API_KEY) > 10, "API key looks empty or broken — check the Colab secret"
print(f"Key loaded OK, length: {len(API_KEY)}")

BASE_URL = "https://api.openaq.org/v3"
headers = {"X-API-Key": API_KEY}

Key loaded OK, length: 64


In [23]:
def safe_get(url, params=None, max_retries=5):
    """GET with automatic backoff on 429 rate-limit responses."""
    for attempt in range(max_retries):
        resp = requests.get(url, headers=headers, params=params)
        if resp.status_code == 429:
            wait = int(resp.headers.get("Retry-After", 0)) or (5 * (attempt + 1))
            print(f"    Rate limited, waiting {wait}s (attempt {attempt+1}/{max_retries})...")
            time.sleep(wait)
            continue
        return resp
    return resp

def find_active_pm25_sensor(lat, lon, radius=25000, max_candidates=40):
    """Find a PM2.5 sensor near lat/lon with the most recent data."""
    resp = safe_get(f"{BASE_URL}/locations", params={"coordinates": f"{lat},{lon}", "radius": radius, "limit": 100})
    if resp.status_code != 200:
        print(f"  Locations error {resp.status_code}: {resp.text[:150]}")
        return None, None, None, None

    locations = resp.json().get("results", [])
    candidates = []
    for loc in locations:
        for sensor in loc.get("sensors", []):
            if sensor.get("parameter", {}).get("name") == "pm25":
                candidates.append((sensor["id"], loc["name"], loc["id"]))

    best, best_last = None, None
    for sensor_id, loc_name, loc_id in candidates[:max_candidates]:
        meta_resp = safe_get(f"{BASE_URL}/sensors/{sensor_id}")
        time.sleep(0.6)
        if meta_resp.status_code != 200:
            continue
        meta = meta_resp.json().get("results", [])
        if not meta:
            continue
        datetime_last_obj = meta[0].get("datetimeLast") or {}
        last_str = datetime_last_obj.get("utc")
        if not last_str:
            continue
        last_dt = datetime.fromisoformat(last_str.replace("Z", "+00:00"))
        if best_last is None or last_dt > best_last:
            best_last, best = last_dt, (sensor_id, loc_name, loc_id)

    if best is None:
        return None, None, None, None
    return best[0], best[1], best[2], best_last

In [28]:
cities = {
    "Jaipur":    (26.9124, 75.7873),
    "Delhi":     (28.6139, 77.2090),
    "Mumbai":    (19.0760, 72.8777),
    "Bengaluru": (12.9716, 77.5946),
    "Kolkata":   (22.5726, 88.3639),
}

city_sensors = {}
for city, (lat, lon) in cities.items():
    sensor_id, loc_name, loc_id, last_seen = find_active_pm25_sensor(lat, lon)
    city_sensors[city] = {"sensor_id": sensor_id, "location_name": loc_name, "location_id": loc_id, "last_seen": last_seen}
    print(f"{city}: sensor_id={sensor_id}, station='{loc_name}', last_data={last_seen}")

Jaipur: sensor_id=12234876, station='Shastri Nagar, Jaipur - RSPCB', last_data=2026-07-06 08:30:00+00:00
Delhi: sensor_id=12234787, station='R K Puram, Delhi - DPCC', last_data=2026-07-06 08:30:00+00:00
Mumbai: sensor_id=12236418, station='Borivali East, Mumbai - IITM', last_data=2026-07-06 08:30:00+00:00
    Rate limited, waiting 5s (attempt 1/5)...
    Rate limited, waiting 10s (attempt 2/5)...
Bengaluru: sensor_id=15203555, station='Koramangala', last_data=2026-07-06 10:00:00+00:00
Kolkata: sensor_id=12234975, station='Padmapukur, Howrah - WBPCB', last_data=2026-07-06 08:30:00+00:00


In [29]:
def fetch_hourly_data(sensor_id, days_back=90):
    """Fetch hourly PM2.5 data for a sensor over the last `days_back` days."""
    date_to = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    date_from = (datetime.now(timezone.utc) - timedelta(days=days_back)).strftime("%Y-%m-%d")

    all_records = []
    page = 1
    while True:
        url = f"{BASE_URL}/sensors/{sensor_id}/hours"
        params = {"datetime_from": date_from, "datetime_to": date_to, "limit": 1000, "page": page}
        resp = requests.get(url, headers=headers, params=params)
        if resp.status_code != 200:
            print(f"  Error {resp.status_code} on page {page}: {resp.text[:150]}")
            break

        results = resp.json().get("results", [])
        if not results:
            break

        for r in results:
            all_records.append({
                "datetime_utc": r["period"]["datetimeFrom"]["utc"],
                "value": r["value"],
                "parameter": r["parameter"]["name"],
                "units": r["parameter"]["units"],
            })

        if len(results) < 1000:
            break
        page += 1
        time.sleep(1)

    return pd.DataFrame(all_records)

In [30]:
all_dfs = []
for city, info in city_sensors.items():
    sensor_id = info["sensor_id"]
    if sensor_id is None:
        print(f"Skipping {city}, no sensor found")
        continue
    print(f"Fetching {city} (sensor {sensor_id})...")
    df = fetch_hourly_data(sensor_id, days_back=90)
    df["city"] = city
    all_dfs.append(df)
    time.sleep(1)

aqi_raw = pd.concat(all_dfs, ignore_index=True)
print(aqi_raw.shape)
aqi_raw.head(10)

Fetching Jaipur (sensor 12234876)...
Fetching Delhi (sensor 12234787)...
Fetching Mumbai (sensor 12236418)...
Fetching Bengaluru (sensor 15203555)...
Fetching Kolkata (sensor 12234975)...
(9773, 5)


,datetime_utc,value,parameter,units,city
0,2026-04-06T19:30:00Z,28.9,pm25,µg/m³,Jaipur
1,2026-04-06T20:30:00Z,34.9,pm25,µg/m³,Jaipur
2,2026-04-06T21:30:00Z,28.0,pm25,µg/m³,Jaipur
3,2026-04-06T22:30:00Z,25.3,pm25,µg/m³,Jaipur
4,2026-04-06T23:30:00Z,27.3,pm25,µg/m³,Jaipur
5,2026-04-07T00:30:00Z,23.7,pm25,µg/m³,Jaipur
6,2026-04-07T01:30:00Z,27.6,pm25,µg/m³,Jaipur
7,2026-04-07T02:30:00Z,35.6,pm25,µg/m³,Jaipur
8,2026-04-07T03:30:00Z,42.8,pm25,µg/m³,Jaipur
9,2026-04-07T04:30:00Z,52.2,pm25,µg/m³,Jaipur


In [31]:
aqi_raw['datetime_utc'] = pd.to_datetime(aqi_raw['datetime_utc'])
aqi_raw = aqi_raw.sort_values(['city', 'datetime_utc']).reset_index(drop=True)
aqi_raw['datetime_ist'] = aqi_raw['datetime_utc'].dt.tz_convert('Asia/Kolkata')

before = len(aqi_raw)
aqi_raw = aqi_raw.drop_duplicates(subset=['city', 'datetime_utc'])
print(f"Dropped {before - len(aqi_raw)} duplicate rows")

print(aqi_raw.groupby('city')['value'].describe())

before = len(aqi_raw)
aqi_raw = aqi_raw[(aqi_raw['value'] >= 0) & (aqi_raw['value'] < 1000)]
print(f"Removed {before - len(aqi_raw)} rows with invalid values")

for city in aqi_raw['city'].unique():
    city_df = aqi_raw[aqi_raw['city'] == city]
    span_hours = (city_df['datetime_utc'].max() - city_df['datetime_utc'].min()).total_seconds() / 3600
    coverage_pct = len(city_df) / span_hours * 100
    print(f"{city}: {len(city_df)} rows over ~{span_hours:.0f}h span ({coverage_pct:.1f}% coverage)")

aqi_clean = aqi_raw[['city', 'datetime_utc', 'datetime_ist', 'value']].rename(columns={'value': 'pm25'})
aqi_clean.to_csv('aqi_clean.csv', index=False)
print(aqi_clean.shape)
aqi_clean.head()

Dropped 0 duplicate rows
            count       mean        std  min   25%   50%     75%    max
city                                                                   
Bengaluru  2135.0  28.490726  16.945791  0.0  13.5  30.0  42.400  192.0
Delhi      1943.0  56.236397  36.583121  1.0  31.0  46.3  70.150  270.0
Jaipur     1836.0  55.923164  23.633748  0.0  41.4  52.2  67.225  507.0
Kolkata    1868.0  19.930086  22.859443  0.0  13.9  18.1  22.600  817.0
Mumbai     1991.0  25.069918  12.731692  0.0  16.0  22.3  31.000   87.4
Removed 0 rows with invalid values
Bengaluru: 2135 rows over ~2159h span (98.9% coverage)
Delhi: 1943 rows over ~2158h span (90.0% coverage)
Jaipur: 1836 rows over ~2158h span (85.1% coverage)
Kolkata: 1868 rows over ~2120h span (88.1% coverage)
Mumbai: 1991 rows over ~2158h span (92.3% coverage)
(9773, 4)


,city,datetime_utc,datetime_ist,pm25
0,Bengaluru,2026-04-06 18:30:00+00:00,2026-04-07 00:00:00+05:30,30.4
1,Bengaluru,2026-04-06 19:30:00+00:00,2026-04-07 01:00:00+05:30,15.2
2,Bengaluru,2026-04-06 20:30:00+00:00,2026-04-07 02:00:00+05:30,8.6
3,Bengaluru,2026-04-06 21:30:00+00:00,2026-04-07 03:00:00+05:30,7.6
4,Bengaluru,2026-04-06 22:30:00+00:00,2026-04-07 04:00:00+05:30,14.2


In [32]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

def build_hourly_grid(df, city):
    city_df = df[df['city'] == city].set_index('datetime_utc').sort_index()
    full_range = pd.date_range(city_df.index.min(), city_df.index.max(), freq='h', tz='UTC')
    city_df = city_df.reindex(full_range)
    city_df['city'] = city
    city_df['pm25'] = city_df['pm25'].interpolate(method='linear', limit=3)
    return city_df

gridded = [build_hourly_grid(aqi_clean, c) for c in aqi_clean['city'].unique()]
aqi_grid = pd.concat(gridded)
aqi_grid.index.name = 'datetime_utc'
aqi_grid = aqi_grid.reset_index()

print(f"Rows before dropping unfillable gaps: {len(aqi_grid)}")
aqi_model = aqi_grid.dropna(subset=['pm25']).copy()
print(f"Rows after: {len(aqi_model)}")
aqi_model['datetime_ist'] = aqi_model['datetime_utc'].dt.tz_convert('Asia/Kolkata')

def add_features(df):
    df = df.sort_values(['city', 'datetime_utc']).copy()
    df['hour'] = df['datetime_ist'].dt.hour
    df['dow'] = df['datetime_ist'].dt.dayofweek
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
    df['lag_1h'] = df.groupby('city')['pm25'].shift(1)
    df['lag_24h'] = df.groupby('city')['pm25'].shift(24)
    df['roll_mean_6h'] = df.groupby('city')['pm25'].transform(lambda s: s.shift(1).rolling(6).mean())
    return df

featured = add_features(aqi_model)
featured = featured.dropna(subset=['lag_1h', 'lag_24h', 'roll_mean_6h']).reset_index(drop=True)

feature_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'lag_1h', 'lag_24h', 'roll_mean_6h']
city_dummies = pd.get_dummies(featured['city'], prefix='city')
X = pd.concat([featured[feature_cols], city_dummies], axis=1)
y = featured['pm25']

sorted_idx = featured.sort_values('datetime_utc').index
split_idx = int(len(sorted_idx) * 0.85)
train_idx, test_idx = sorted_idx[:split_idx], sorted_idx[split_idx:]

X_train, X_test = X.loc[train_idx], X.loc[test_idx]
y_train, y_test = y.loc[train_idx], y.loc[test_idx]

model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
print(f"Test MAE: {mae:.2f} µg/m³ (on {len(X_test)} held-out hourly points)")

Rows before dropping unfillable gaps: 10758
Rows after: 10184
Test MAE: 6.33 µg/m³ (on 1510 held-out hourly points)


In [33]:
CPCB_BREAKPOINTS = [
    (0, 30, "Good", "Safe for all outdoor activity, including sensitive groups."),
    (30, 60, "Satisfactory", "Generally safe. Sensitive individuals may notice minor discomfort on prolonged exertion."),
    (60, 90, "Moderate", "Reduce prolonged outdoor exertion if you have asthma, heart, or lung conditions."),
    (90, 120, "Poor", "Limit outdoor activity, especially for children, elderly, and those with respiratory conditions."),
    (120, 250, "Very Poor", "Avoid outdoor exercise. Sensitive groups should stay indoors where possible."),
    (250, float('inf'), "Severe", "Avoid all outdoor activity. Health risk even for healthy individuals."),
]

def categorize_pm25(value):
    for lo, hi, label, advice in CPCB_BREAKPOINTS:
        if lo <= value < hi:
            return label, advice
    return "Severe", CPCB_BREAKPOINTS[-1][2]

def recursive_forecast(city, hist_df, model, feature_cols, city_dummy_cols, hours_ahead=24):
    city_hist = hist_df[hist_df['city'] == city].sort_values('datetime_utc').copy()
    recent = city_hist['pm25'].tolist()
    last_time = city_hist['datetime_ist'].iloc[-1]

    forecasts = []
    for step in range(1, hours_ahead + 1):
        next_time = last_time + pd.Timedelta(hours=step)
        hour, dow = next_time.hour, next_time.dayofweek

        lag_1h = recent[-1]
        lag_24h = recent[-24] if len(recent) >= 24 else recent[0]
        roll_mean_6h = np.mean(recent[-6:])

        row = {
            'hour_sin': np.sin(2 * np.pi * hour / 24), 'hour_cos': np.cos(2 * np.pi * hour / 24),
            'dow_sin': np.sin(2 * np.pi * dow / 7), 'dow_cos': np.cos(2 * np.pi * dow / 7),
            'lag_1h': lag_1h, 'lag_24h': lag_24h, 'roll_mean_6h': roll_mean_6h,
        }
        for col in city_dummy_cols:
            row[col] = 1 if col == f'city_{city}' else 0

        X_next = pd.DataFrame([row])[feature_cols + city_dummy_cols]
        pred = max(model.predict(X_next)[0], 0)

        recent.append(pred)
        category, advice = categorize_pm25(pred)
        forecasts.append({'city': city, 'datetime_ist': next_time, 'hours_ahead': step,
                           'pm25_forecast': round(pred, 1), 'category': category, 'advice': advice})

    return pd.DataFrame(forecasts)

In [34]:
city_dummy_cols = list(city_dummies.columns)
now_ist = pd.Timestamp.now(tz='Asia/Kolkata')

all_forecasts = []
for city in featured['city'].unique():
    city_hist = featured[featured['city'] == city]
    last_time = city_hist['datetime_ist'].max()
    gap_hours = int((now_ist - last_time).total_seconds() / 3600)
    hours_needed = max(gap_hours, 0) + 24

    fc = recursive_forecast(city, featured, model, feature_cols, city_dummy_cols, hours_ahead=hours_needed)
    fc_future = fc[fc['datetime_ist'] >= now_ist].reset_index(drop=True)
    all_forecasts.append(fc_future)
    print(f"{city}: gap={gap_hours}h, kept {len(fc_future)} future rows")

forecast_df = pd.concat(all_forecasts, ignore_index=True)
forecast_df.to_csv('forecast_24h.csv', index=False)

for city in forecast_df['city'].unique():
    print(f"\n{city} — next 6 hours from now:")
    print(forecast_df[forecast_df['city'] == city].head(6)[['datetime_ist', 'pm25_forecast', 'category']].to_string(index=False))

Bengaluru: gap=17h, kept 24 future rows
Delhi: gap=17h, kept 24 future rows
Jaipur: gap=17h, kept 24 future rows
Kolkata: gap=55h, kept 24 future rows
Mumbai: gap=17h, kept 24 future rows

Bengaluru — next 6 hours from now:
             datetime_ist  pm25_forecast     category
2026-07-06 17:00:00+05:30           37.7 Satisfactory
2026-07-06 18:00:00+05:30           38.5 Satisfactory
2026-07-06 19:00:00+05:30           37.8 Satisfactory
2026-07-06 20:00:00+05:30           40.6 Satisfactory
2026-07-06 21:00:00+05:30           40.4 Satisfactory
2026-07-06 22:00:00+05:30           39.3 Satisfactory

Delhi — next 6 hours from now:
             datetime_ist  pm25_forecast     category
2026-07-06 17:00:00+05:30           37.0 Satisfactory
2026-07-06 18:00:00+05:30           38.1 Satisfactory
2026-07-06 19:00:00+05:30           43.1 Satisfactory
2026-07-06 20:00:00+05:30           46.3 Satisfactory
2026-07-06 21:00:00+05:30           46.3 Satisfactory
2026-07-06 22:00:00+05:30           46.5 S

In [35]:
!pip install -q google-genai

from google import genai

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY').strip()
client = genai.Client(api_key=GEMINI_API_KEY)

# Quick sanity check
test = client.models.generate_content(model="gemini-2.5-flash", contents="Say OK if you're working.")
print(test.text)

OK


In [36]:
def ask_aqi_assistant(question, city, forecast_df, hours_context=12):
    """Answer a natural-language question using the city's real forecast data as grounding."""
    city_data = forecast_df[forecast_df['city'].str.lower() == city.lower()].head(hours_context)
    if city_data.empty:
        return f"I don't have forecast data for '{city}'. Available cities: {', '.join(forecast_df['city'].unique())}"

    data_summary = city_data[['datetime_ist', 'pm25_forecast', 'category']].to_string(index=False)

    prompt = f"""You are an air quality advisor for Indian cities. Answer the user's question using ONLY the forecast data below — do not invent numbers.

City: {city}
Forecast for the next {hours_context} hours (PM2.5 in µg/m³, IST):
{data_summary}

CPCB categories: Good (0-30), Satisfactory (30-60), Moderate (60-90), Poor (90-120), Very Poor (120-250), Severe (250+).

User question: {question}

Give a direct, practical answer in 2-4 sentences. Mention the specific PM2.5 values and times that support your answer. If the question is about an activity (running, kids playing outside, elderly walk), give a clear go/wait/avoid recommendation."""

    response = client.models.generate_content(model="gemini-2.5-flash", contents=prompt)
    return response.text

# Test it
answer = ask_aqi_assistant("Is it safe for my mother to go for an evening walk today?", "Jaipur", forecast_df)
print(answer)

For an evening walk today in Jaipur, the air quality will be in the Satisfactory category initially, with PM2.5 at 50.4 µg/m³ at 17:00 IST and 57.0 µg/m³ at 18:00 IST. However, it is forecasted to transition to Moderate, reaching 63.5 µg/m³ by 19:00 IST and peaking at 66.0 µg/m³ around 20:00 IST. Given the Moderate air quality expected during prime evening hours, it would be advisable for your mother to avoid an evening walk or keep it very short if walking before 19:00 IST.


In [37]:
from google.colab import files
files.download('aqi_clean.csv')
files.download('forecast_24h.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

PROJECT_ID = "aqi-decision-assistant"  # the one you created earlier — check console.cloud.google.com, top bar shows it
client = bigquery.Client(project=PROJECT_ID)

# Create a dataset to hold our tables
dataset_id = f"{PROJECT_ID}.aqi_data"
client.create_dataset(dataset_id, exists_ok=True)
print(f"Dataset ready: {dataset_id}")

# Load cleaned historical data
job1 = client.load_table_from_dataframe(aqi_clean, f"{dataset_id}.pm25_hourly_history")
job1.result()
print(f"Loaded {job1.output_rows} rows into pm25_hourly_history")

# Load the forecast too
job2 = client.load_table_from_dataframe(forecast_df, f"{dataset_id}.pm25_forecast_24h")
job2.result()
print(f"Loaded {job2.output_rows} rows into pm25_forecast_24h")

# Quick proof-of-life query — this is a good screenshot for your deck's architecture slide
query = f"""
SELECT city, ROUND(AVG(pm25), 1) as avg_pm25, MAX(pm25) as peak_pm25
FROM `{dataset_id}.pm25_hourly_history`
GROUP BY city
ORDER BY avg_pm25 DESC
"""
result = client.query(query).to_dataframe()
print(result)

Dataset ready: aqi-decision-assistant.aqi_data
Loaded 9773 rows into pm25_hourly_history
Loaded 120 rows into pm25_forecast_24h
        city  avg_pm25  peak_pm25
0      Delhi      56.2      270.0
1     Jaipur      55.9      507.0
2  Bengaluru      28.5      192.0
3     Mumbai      25.1       87.4
4    Kolkata      19.9      817.0


In [40]:
from google.colab import files
files.download('aqi_clean.csv')
files.download('forecast_24h.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>